# Tabular Data Classification

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader 
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

In [5]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

In [6]:
df = pd.read_csv('/Users/atharvakhandelwal/Desktop/learningpandas/10_Pytorch/data/riceClassification.csv')
df.head(3)

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1


In [7]:
df = df.drop(columns=['id'], axis=1)
df.dropna(inplace=True)
df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [8]:
print(df['Class'].unique())
print('-'*100)
print(df['Class'].value_counts())

[1 0]
----------------------------------------------------------------------------------------------------
Class
1    9985
0    8200
Name: count, dtype: int64


In [9]:
original_df = df.copy()

In [10]:
for column in df.columns:
    df[column] = df[column]/ df[column].max()

In [11]:
df

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,0.444368,0.503404,0.775435,0.744658,0.424873,0.666610,0.741661,0.537029,0.844997,0.368316,1.0
1,0.281293,0.407681,0.622653,0.750489,0.273892,0.530370,0.804230,0.409661,0.919215,0.371471,1.0
2,0.298531,0.416421,0.630442,0.756341,0.284520,0.546380,0.856278,0.412994,0.959862,0.374747,1.0
3,0.300979,0.420463,0.629049,0.764024,0.286791,0.548616,0.883772,0.414262,0.961818,0.379222,1.0
4,0.361704,0.464626,0.682901,0.775033,0.345385,0.601418,0.867808,0.452954,0.966836,0.386007,1.0
...,...,...,...,...,...,...,...,...,...,...,...
18180,0.573262,0.811219,0.618156,0.971489,0.545785,0.757140,0.562384,0.654774,0.733291,0.744543,0.0
18181,0.742899,0.925674,0.704314,0.971683,0.709121,0.861916,0.730296,0.758107,0.708884,0.745661,0.0
18182,0.623408,0.844800,0.640916,0.972058,0.593296,0.789562,0.633098,0.673049,0.754720,0.747830,0.0
18183,0.583741,0.826356,0.623551,0.972748,0.562227,0.764030,0.555396,0.675248,0.702103,0.751874,0.0


In [12]:
X = np.array(df.iloc[:,:-1])
y = np.array(df.iloc[:,-1])

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)

In [14]:
X_test, X_val, y_test, y_val = train_test_split(X_test,y_test, test_size=0.5, random_state=42)

In [15]:
class dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y, dtype=torch.long).to(device)

    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [16]:
training_dataset = dataset(X_train, y_train)
validation_dataset = dataset(X_val, y_val)
test_dataset = dataset(X_test, y_test)

In [17]:
train_loader = DataLoader(training_dataset, batch_size=64, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [18]:
h_n = 10
input_ft = X.shape[1]

class Mynn(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Linear(input_ft, 16), # Increased neurons slightly
            nn.ReLU(),                     # Added Activation Function!
            nn.Linear(16, 16),             # Added one more layer for better learning
            nn.ReLU(),
            nn.Linear(16, 1))

    def forward(self, x):
        return self.layer_stack(x)

In [19]:
model = Mynn().to(device)

In [26]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [27]:
# for data, label in train_loader:
#     print(data.shape)
#     print(label.shape)
#     break

In [28]:
epochs = 50

for epoch in range(epochs):
    model.train() 
    train_loss = 0
    train_acc = 0
    
    for X_batch, y_batch in train_loader:
        y_batch = y_batch.unsqueeze(1).float()
        y_pred_logits = model(X_batch)
        loss = criterion(y_pred_logits, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step() 

        train_loss += loss.item()
        
        y_pred_tag = torch.round(torch.sigmoid(y_pred_logits))
        train_acc += (y_pred_tag == y_batch).sum().item()

    avg_loss = train_loss / len(train_loader)
    avg_acc = train_acc / len(X_train)
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: | Loss: {avg_loss:.5f} | Acc: {avg_acc*100:.2f}%')

Epoch 10: | Loss: 0.04184 | Acc: 98.60%
Epoch 20: | Loss: 0.04067 | Acc: 98.76%
Epoch 30: | Loss: 0.03967 | Acc: 98.69%
Epoch 40: | Loss: 0.03932 | Acc: 98.70%
Epoch 50: | Loss: 0.03855 | Acc: 98.78%


In [30]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for X_b, y_b in validation_loader:
        X_b = X_b.to(device)
        y_b = y_b.to(device)
        
        y_b = y_b.unsqueeze(1) 
        
        y_pred = model(X_b)
        y_pred_tag = torch.round(torch.sigmoid(y_pred))
        
        correct += (y_pred_tag == y_b).sum().item()
        total += y_b.shape[0]
    
    print(f"\nValidation Accuracy: {correct/total*100:.2f}%")


Validation Accuracy: 98.42%


In [31]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0

    for X_t, y_t in test_loader:
        X_t = X_t.to(device)
        y_t = y_t.to(device)

        y_t = y_t.unsqueeze(1) 

        y_pred = model(X_t)
        y_pred_tag = torch.round(torch.sigmoid(y_pred))
        
        correct += (y_pred_tag == y_t).sum().item()
        total += y_t.shape[0]
    
    print(f"\nValidation Accuracy: {correct/total*100:.2f}%")




Validation Accuracy: 99.27%
